In [1]:
# ── Cell 1: System + Python deps ─────────────────────────────────────────────
# Kokoro needs espeak-ng (system) + its pip package.
# All other deps remain the same as before.
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install -q -U diffusers transformers accelerate sentencepiece soundfile tqdm bitsandbytes "kokoro>=0.9.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.1/516.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import os, gc, zipfile, urllib.request, shutil, re, math
import pandas as pd
import numpy as np
import torch
import soundfile as sf
from PIL import Image
from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig
from tqdm import tqdm
from kokoro import KPipeline   # Kokoro TTS — clean, vibrant, zero filler words

# ============================================================
#  USER CONFIGURATION  ← edit these before running
# ============================================================
ASPECT_RATIO   = "16:9"        # "16:9" or "9:16"
BGM_URL        = "https://docs.google.com/uc?export=download&id=1OkWbdEFlh3N4kcl7zazUj_N0X9VzB6IP"
CSV_DATA_URL   = "https://docs.google.com/spreadsheets/d/1_9vG-_5RUmPJFNl-d8D38lvRpRo550AUabL-fnMnwtE/export?format=csv"
USE_Z_IMAGE    = False
Z_IMAGE_KEY    = ""

# ── Voice ────────────────────────────────────────────────────
# American English vibrant female voices: af_heart ✨ af_nova  af_sky  af_bella
# American English vibrant male voices  : am_echo   am_onyx   am_michael
KOKORO_VOICE   = "am_michael"    # energetic, expressive female — great for content

# ── Captions ────────────────────────────────────────────────
CAPTION_ENABLED   = True       # burn 4-word captions onto the video
CAPTION_FONT_SIZE = 22         # pixel height — increase for bigger text (e.g. 90)
CAPTION_COLOR     = "white"    # text fill colour
CAPTION_OUTLINE   = 4          # border thickness in pixels (0 = no outline)
CAPTION_Y_POS     = 0.80       # vertical position as fraction of height (0=top 1=bottom)

# ── Speed ────────────────────────────────────────────────────
# 1.0 = normal | 1.1 = 10% faster | 1.25 = 25% faster | 0.9 = 10% slower
# Range: 0.5 – 2.0  (values outside that range are clamped automatically)
VIDEO_SPEED    = 1.1

# ── Output ──────────────────────────────────────────────────
FPS            = 24

# ── HuggingFace Token (PASTE YOURS HERE — no Secrets needed) ─
# Get a READ token from https://huggingface.co/settings/tokens
# Also accept the Flux licence at:
# https://huggingface.co/black-forest-labs/FLUX.1-schnell
HF_TOKEN_OVERRIDE = ""    # ← paste your hf_... token between the quotes
# ============================================================

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if ASPECT_RATIO == "16:9":
    WIDTH, HEIGHT = 736, 416
elif ASPECT_RATIO == "9:16":
    WIDTH, HEIGHT = 416, 736
else:
    raise ValueError("ASPECT_RATIO must be '16:9' or '9:16'")

print(f"--> Settings: {ASPECT_RATIO} | {WIDTH}x{HEIGHT} | FPS:{FPS} | Speed:{VIDEO_SPEED}x")
print(f"    Voice: {KOKORO_VOICE} | Captions: {CAPTION_ENABLED} | Font size: {CAPTION_FONT_SIZE}px")

# ── HuggingFace auth — LAZY (only called if Flux images are missing) ──────────
# This never crashes at startup. The token is only fetched + validated
# at the moment Flux.from_pretrained() is actually called.
# If all images already exist on disk, this block is never executed.

def _ensure_hf_auth():
    """
    Authenticate with HuggingFace. Called only when Flux needs to load.
    Tries 3 sources in order:
      1. HF_TOKEN_OVERRIDE pasted directly in the config block above  ← most reliable
      2. Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
      3. HF_TOKEN environment variable
    """
    global HF_TOKEN
    if HF_TOKEN:
        return HF_TOKEN   # already authenticated this session

    from huggingface_hub import login as hf_login, whoami

    candidates = []

    # Priority 1: hardcoded override in config (always works, no platform magic)
    if HF_TOKEN_OVERRIDE and HF_TOKEN_OVERRIDE.strip().startswith("hf_"):
        candidates.append(("config override", HF_TOKEN_OVERRIDE.strip()))

    # Priority 2: Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient().get_secret("HF_TOKEN")
        if secret and secret.strip().startswith("hf_"):
            candidates.append(("Kaggle Secret", secret.strip()))
    except Exception as e:
        print(f"  [INFO] Kaggle Secrets not available ({type(e).__name__}) — that is OK.")

    # Priority 3: environment variable
    import os as _os
    env_tok = _os.environ.get("HF_TOKEN", "").strip()
    if env_tok.startswith("hf_"):
        candidates.append(("environment variable", env_tok))

    for source, token in candidates:
        try:
            hf_login(token=token, add_to_git_credential=False)
            HF_TOKEN = token
            print(f"--> HF authenticated via {source} as: {whoami()['name']}")
            return HF_TOKEN
        except Exception as e:
            print(f"  [WARN] Token from {source} rejected: {e}")

    # Nothing worked
    raise RuntimeError(
        "\n" + "="*60 + "\n"
        "  HF_TOKEN NOT FOUND — Flux cannot load without it.\n"
        "="*60 + "\n\n"
        "QUICKEST FIX (paste directly in notebook):\n"
        "  1. Accept Flux licence:\n"
        "     https://huggingface.co/black-forest-labs/FLUX.1-schnell\n"
        "  2. Create a READ token:\n"
        "     https://huggingface.co/settings/tokens\n"
        "  3. In the USER CONFIGURATION section at the top of this notebook,\n"
        "     set:  HF_TOKEN_OVERRIDE = \"hf_your_token_here\"\n"
        "  4. Save & Run All again.\n\n"
        "That's it — no Secrets, no env vars, no toggles needed."
    )

# ── Output directories ────────────────────────────────────────────────────────
OUTPUT_DIR  = "/kaggle/working/automated_channel_outputs"
IMAGES_DIR  = os.path.join(OUTPUT_DIR, "images")
EFFECT_DIR  = os.path.join(OUTPUT_DIR, "effect_videos")
AUDIO_DIR   = os.path.join(OUTPUT_DIR, "audios")
STITCH_DIR  = os.path.join(OUTPUT_DIR, "stitched")
for d in [IMAGES_DIR, EFFECT_DIR, AUDIO_DIR, STITCH_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Restore images from previous Kaggle output version (if attached) ──────────
# When a previous commit generated images, Kaggle saves them under
# /kaggle/input/<notebook-slug>/<version>/images/
# If you attach the previous output as an input dataset, images are auto-restored
# here so Flux is skipped and the token is never needed.
_PREV_OUTPUT_ROOTS = [
    "/kaggle/input/automated-channel-outputs",          # common slug
    "/kaggle/input/automated_channel_outputs",
]
_restored = 0
for _root in _PREV_OUTPUT_ROOTS:
    _prev_img = os.path.join(_root, "images")
    if os.path.isdir(_prev_img):
        for _f in os.listdir(_prev_img):
            _src = os.path.join(_prev_img, _f)
            _dst = os.path.join(IMAGES_DIR, _f)
            if not os.path.exists(_dst):
                shutil.copy2(_src, _dst)
                _restored += 1
        if _restored:
            print(f"--> Restored {_restored} image(s) from previous output: {_prev_img}")
        break

# ── Helpers ───────────────────────────────────────────────────────────────────
def force_clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    gc.collect()

def all_outputs_exist(directory, serials, ext):
    return all(os.path.exists(os.path.join(directory, f"{sn}{ext}")) for sn in serials)

def _safe_text_for_drawtext(text):
    """Escape characters that break FFmpeg drawtext expressions."""
    text = text.replace("\\", "\\\\")
    text = text.replace("'",  "\u2019")   # curly apostrophe — avoids shell quoting hell
    text = text.replace(":",  "\\:")
    text = text.replace(",",  "\\,")
    text = text.replace("[",  "\\[").replace("]", "\\]")
    return text

def build_caption_filter(voiceover_text, duration, font_size, W, H):
    """
    Splits voiceover into 4-word chunks and builds an FFmpeg drawtext filter
    that displays each chunk for its proportional time slice.

    Returns a ready-to-use vf segment string (or 'null' if nothing to do).
    """
    words = voiceover_text.split()
    if not words or not CAPTION_ENABLED:
        return "null"

    groups = [' '.join(words[i:i+4]) for i in range(0, len(words), 4)]
    seg_dur = duration / len(groups)
    parts = []

    for i, group in enumerate(groups):
        t0 = round(i * seg_dur, 4)
        t1 = round((i + 1) * seg_dur, 4)
        safe = _safe_text_for_drawtext(group)
        y_expr = f"h*{CAPTION_Y_POS:.3f}-text_h/2"
        part = (
            f"drawtext=text='{safe}':"
            f"enable='between(t,{t0},{t1})':"
            f"fontsize={font_size}:"
            f"fontcolor={CAPTION_COLOR}:"
            f"borderw={CAPTION_OUTLINE}:bordercolor=black:"
            f"x=(w-text_w)/2:"
            f"y={y_expr}:"
            f"font=DejaVuSans-Bold"
        )
        parts.append(part)

    return ",".join(parts)

def build_atempo_chain(speed):
    """
    atempo only accepts 0.5–2.0. For values outside that, chain filters.
    e.g. 2.5x = atempo=2.0,atempo=1.25
    """
    speed = max(0.25, min(4.0, speed))   # hard clamp
    filters = []
    remaining = speed
    while remaining > 2.0:
        filters.append("atempo=2.0")
        remaining /= 2.0
    while remaining < 0.5:
        filters.append("atempo=0.5")
        remaining /= 0.5
    filters.append(f"atempo={remaining:.6f}")
    return ",".join(filters)

# ── Load input CSV ────────────────────────────────────────────────────────────
print("\n--> Loading input dataset...")
try:
    df = pd.read_csv(CSV_DATA_URL, storage_options={"User-Agent": "Mozilla/5.0"})
    # Normalize columns case-insensitively
    col_map = {c.lower().strip().replace('_', ' '): c for c in df.columns}
    rename_map = {}
    if 'serial number' in col_map: rename_map[col_map['serial number']] = 'Serial number'
    elif 'serial no' in col_map: rename_map[col_map['serial no']] = 'Serial number'
    elif 'serial' in col_map: rename_map[col_map['serial']] = 'Serial number'
    if 'image prompt' in col_map: rename_map[col_map['image prompt']] = 'image prompt'
    elif 'image' in col_map: rename_map[col_map['image']] = 'image prompt'
    if 'voice over prompt' in col_map: rename_map[col_map['voice over prompt']] = 'voice over prompt'
    elif 'voiceover prompt' in col_map: rename_map[col_map['voiceover prompt']] = 'voice over prompt'
    elif 'voice over' in col_map: rename_map[col_map['voice over']] = 'voice over prompt'
    elif 'voiceover' in col_map: rename_map[col_map['voiceover']] = 'voice over prompt'
    df.rename(columns=rename_map, inplace=True)
    df.to_csv(os.path.join(OUTPUT_DIR, "downloaded_manifest.csv"), index=False)
    print(f"    Loaded from Google Drive: {len(df)} entries.")
except Exception as e:
    fallback_paths = [
        "/kaggle/input/channel-data/input_data.csv",
        "/kaggle/input/input-data/input_data.csv",
    ]
    loaded = False
    for fp in fallback_paths:
        if os.path.exists(fp):
            df = pd.read_csv(fp)
            print(f"    Loaded from local dataset ({fp}): {len(df)} entries.")
            loaded = True
            break
    if not loaded:
        raise RuntimeError(f"Failed to load CSV: {e}")

if "audio_length" not in df.columns:
    df["audio_length"] = 0.0
df["audio_length"] = df["audio_length"].astype(float)

# Shared 4-bit quantization config for Flux
QUANT_CFG = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": torch.bfloat16,
    },
    components_to_quantize=["transformer", "text_encoder_2"],
)


# ============================================================
# PHASE 1 — Flux.1-Schnell  (Image Generation)
# ============================================================

# ============================================================
# PHASE 1B — Z Image API (kie.ai)
# ============================================================
def run_phase_1_z_image():
    import requests, time, urllib.request
    print("\n" + "="*60)
    print("PHASE 1: Image Generation (Z Image API)")
    print("="*60)
    
    if not Z_IMAGE_KEY:
        raise ValueError("Z_IMAGE_KEY is missing but USE_Z_IMAGE is True.")
    print(f"  API Key: {Z_IMAGE_KEY[:8]}...{Z_IMAGE_KEY[-4:]}")

    headers = {
        "Authorization": f"Bearer {Z_IMAGE_KEY}",
        "Content-Type": "application/json"
    }
    
    serials = [str(r["Serial number"]) for _, r in df.iterrows()]
    if all_outputs_exist(IMAGES_DIR, serials, ".png"):
        print("--> All images already on disk. Skipping Z Image generation.")
        return

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Rendering via Z Image"):
        sn = str(row["Serial number"])
        img_path = os.path.join(IMAGES_DIR, f"{sn}.png")
        if os.path.exists(img_path):
            continue

        prompt = str(row["image prompt"])[:120]
        print(f"\n  [{sn}] Prompt: {prompt}...")
        
        # 1. Create task
        create_payload = {
            "model": "z-image",
            "input": {
                "prompt": str(row["image prompt"]),
                "aspect_ratio": ASPECT_RATIO,
                "nsfw_checker": False
            }
        }
        print(f"  Sending createTask to kie.ai...")
        
        max_retries = 3
        task_id = None
        for attempt in range(max_retries):
            try:
                resp = requests.post("https://api.kie.ai/api/v1/jobs/createTask", json=create_payload, headers=headers, timeout=30)
                print(f"  Response status: {resp.status_code}")
                print(f"  Response body: {resp.text[:500]}")
                resp.raise_for_status()
                data = resp.json()
                if data.get("code") == 200:
                    task_id = data.get("data", {}).get("taskId")
                    print(f"  Task created: {task_id}")
                    break
                else:
                    print(f"  [WARN] Attempt {attempt+1}: API returned code {data.get('code')}: {data}")
            except Exception as e:
                print(f"  [WARN] Attempt {attempt+1}: Request failed: {type(e).__name__}: {e}")
            time.sleep(3)
            
        if not task_id:
            print(f"  [ERROR] Failed to create task for {sn}. Skipping.")
            continue
            
        # 2. Poll status
        print(f"  Polling for result (max 5 min)...")
        success = False
        for poll_num in range(60):
            time.sleep(5)
            try:
                poll_resp = requests.get(f"https://api.kie.ai/api/v1/jobs/recordInfo?taskId={task_id}", headers=headers, timeout=15)
                poll_data = poll_resp.json()
                state = poll_data.get("data", {}).get("state", "unknown")
                if poll_num % 6 == 0:
                    print(f"    Poll #{poll_num+1}: state={state}")
                if poll_data.get("code") == 200:
                    if state == "success":
                        res_json = json.loads(poll_data.get("data", {}).get("resultJson", "{}"))
                        urls = res_json.get("resultUrls", [])
                        if urls:
                            urllib.request.urlretrieve(urls[0], img_path)
                            print(f"  Saved {sn}.png")
                            success = True
                        else:
                            print(f"  [ERROR] Success but no URLs: {res_json}")
                        break
                    elif state == "fail":
                        print(f"  [ERROR] Task failed for {sn}: {poll_data.get('data', {}).get('failMsg')}")
                        break
                else:
                    print(f"  [WARN] Poll returned code {poll_data.get('code')}: {poll_data}")
            except Exception as e:
                print(f"  [WARN] Poll error: {type(e).__name__}: {e}")
                
        if not success:
            print(f"  [ERROR] Could not fetch image for {sn} after {poll_num+1} polls.")

    # Zip output
    zip_path = os.path.join(OUTPUT_DIR, "source_images.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for f_name in os.listdir(IMAGES_DIR):
            z.write(os.path.join(IMAGES_DIR, f_name), arcname=f_name)
    print("--> Phase 1 Z Image complete.")

def run_phase_1_flux():
    print("\n" + "="*60)
    print("PHASE 1: Image Generation  (Flux.1-Schnell)")
    print("="*60)
    serials = [str(r["Serial number"]) for _, r in df.iterrows()]
    if all_outputs_exist(IMAGES_DIR, serials, ".png"):
        print("--> All images already on disk. Skipping Flux load.")
        return

    token = _ensure_hf_auth()   # fetch+validate token only at this moment
    flux_pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-schnell",
        quantization_config=QUANT_CFG,
        torch_dtype=torch.bfloat16,
        token=token,
    )
    flux_pipe.enable_model_cpu_offload()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Rendering Images"):
        sn       = str(row["Serial number"])
        img_path = os.path.join(IMAGES_DIR, f"{sn}.png")
        if os.path.exists(img_path):
            continue
        image = flux_pipe(
            prompt=str(row["image prompt"]),
            guidance_scale=0.0,
            num_inference_steps=4,
            width=WIDTH,
            height=HEIGHT,
            max_sequence_length=256,
        ).images[0]
        image.save(img_path)
        print(f"  Saved {sn}.png")

    zip_path = os.path.join(OUTPUT_DIR, "source_images.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for f in os.listdir(IMAGES_DIR):
            z.write(os.path.join(IMAGES_DIR, f), arcname=f)

    del flux_pipe
    force_clear_memory()
    print("--> Phase 1 complete. Flux purged from VRAM.")


# ============================================================
# PHASE 2 — Kokoro TTS  (Voice Synthesis)
#
# WHY KOKORO OVER BARK:
#   • Bark is a generative audio LM — it literally "thinks"
#     in tokens, producing filler sounds (ermm, uhh) as
#     natural token output. There is no clean fix for this.
#   • Kokoro is a proper neural TTS (flow-matching).
#     It takes text → speech directly, no generation noise,
#     no filler words, consistent pace.
#   • 82M params, <2GB VRAM, ~10-40x faster than Bark.
#   • Voices like af_heart / af_nova are bright and energetic.
#   • Output is clean float32 numpy — no dtype cast needed.
# ============================================================
KOKORO_SR = 24_000   # Kokoro always outputs 24 kHz

def run_phase_2_audio():
    print("\n" + "="*60)
    print("PHASE 2: Voice Synthesis  (Kokoro TTS)")
    print(f"         Voice: {KOKORO_VOICE}  |  Sample rate: {KOKORO_SR} Hz")
    print("="*60)

    serials = [str(r["Serial number"]) for _, r in df.iterrows()]

    if all_outputs_exist(AUDIO_DIR, serials, ".wav"):
        print("--> All audio already on disk. Reading durations...")
        for idx, row in df.iterrows():
            sn = str(row["Serial number"])
            ap = os.path.join(AUDIO_DIR, f"{sn}.wav")
            if os.path.exists(ap):
                data, sr = sf.read(ap)
                df.at[idx, "audio_length"] = round(len(data) / sr, 3)
        df.to_csv(os.path.join(OUTPUT_DIR, "updated_manifest.csv"), index=False)
        return

    # Kokoro runs fine on CPU but is much faster on GPU.
    # No HF token required — weights are Apache-licensed.
    tts = KPipeline(lang_code="a")   # 'a' = American English

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Kokoro TTS"):
        sn         = str(row["Serial number"])
        audio_path = os.path.join(AUDIO_DIR, f"{sn}.wav")

        if os.path.exists(audio_path):
            data, sr = sf.read(audio_path)
            df.at[idx, "audio_length"] = round(len(data) / sr, 3)
            print(f"  Skipping {sn}.wav (exists, {df.at[idx, 'audio_length']:.1f}s)")
            continue

        voiceover_text = str(row["voice over prompt"])

        # Kokoro generates in chunks; concatenate all chunks into one array.
        chunks = []
        for _, _, audio_chunk in tts(voiceover_text, voice=KOKORO_VOICE):
            if audio_chunk is not None and len(audio_chunk) > 0:
                chunks.append(audio_chunk)

        if not chunks:
            print(f"  WARNING: Kokoro produced no audio for serial {sn}. Skipping.")
            continue

        audio_array = np.concatenate(chunks).astype(np.float32)
        duration    = round(len(audio_array) / KOKORO_SR, 3)
        df.at[idx, "audio_length"] = duration

        sf.write(audio_path, audio_array, KOKORO_SR)
        print(f"  Saved {sn}.wav ({duration:.1f}s)")

    df.to_csv(os.path.join(OUTPUT_DIR, "updated_manifest.csv"), index=False)
    # Kokoro is lightweight — no explicit delete needed, but clear for tidiness
    del tts
    force_clear_memory()
    print("--> Phase 2 complete.")


# ============================================================
# PHASE 3 — FFmpeg Cinematic Motion Effects on Images
#
# 6 rotating Ken Burns / motion effects (no GPU required):
#   0: Slow zoom in   (center)
#   1: Slow zoom out  (center)
#   2: Pan left → right
#   3: Pan right → left
#   4: Diagonal drift + zoom
#   5: Vertical pan bottom → top
#
# Each clip is generated to audio_length + 0.5 s buffer.
# 4-word captions are burned in here at the known timing.
# ============================================================
EFFECT_NAMES = ["zoom-in", "zoom-out", "pan-L→R", "pan-R→L", "diagonal", "vert-pan"]

def _build_vf_motion(effect_idx, frames, W, H):
    f = max(frames, 1)
    effects = [
        # 0: zoom in
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.0+0.5*on/{f}':d={f}:"
        f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H},setsar=1",
        # 1: zoom out
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.5-0.5*on/{f}':d={f}:"
        f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H},setsar=1",
        # 2: pan L→R
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.3':d={f}:"
        f"x='(iw-iw/zoom)*on/{f}':y='ih/2-(ih/zoom/2)':s={W}x{H},setsar=1",
        # 3: pan R→L
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.3':d={f}:"
        f"x='(iw-iw/zoom)*(1-on/{f})':y='ih/2-(ih/zoom/2)':s={W}x{H},setsar=1",
        # 4: diagonal + zoom
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.2+0.2*on/{f}':d={f}:"
        f"x='(iw-iw/zoom)*(1-on/{f})/2':y='(ih-ih/zoom)*(1-on/{f})/2':s={W}x{H},setsar=1",
        # 5: vertical pan
        f"scale=iw*2:ih*2,"
        f"zoompan=z='1.25':d={f}:"
        f"x='iw/2-(iw/zoom/2)':y='(ih-ih/zoom)*(1-on/{f})':s={W}x{H},setsar=1",
    ]
    return effects[effect_idx % len(effects)]


def run_phase_3_effects():
    print("\n" + "="*60)
    print("PHASE 3: Cinematic Motion Effects  (FFmpeg/CPU)")
    print(f"         Captions: {CAPTION_ENABLED} | Font size: {CAPTION_FONT_SIZE}px")
    print("="*60)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Motion + Captions"):
        sn         = str(row["Serial number"])
        img_path   = os.path.join(IMAGES_DIR, f"{sn}.png")
        effect_out = os.path.join(EFFECT_DIR,  f"{sn}.mp4")

        if os.path.exists(effect_out):
            print(f"  Skipping {sn}.mp4 (exists)")
            continue

        if not os.path.exists(img_path):
            print(f"  WARNING: {sn}.png missing — skipping.")
            continue

        audio_len = float(df.at[idx, "audio_length"])
        if audio_len <= 0:
            audio_len = 12.0

        duration = audio_len + 0.5   # small buffer
        frames   = int(duration * FPS)

        motion_vf  = _build_vf_motion(idx, frames, WIDTH, HEIGHT)
        voiceover  = str(row.get("voice over prompt", ""))
        caption_vf = build_caption_filter(voiceover, duration, CAPTION_FONT_SIZE, WIDTH, HEIGHT)

        if CAPTION_ENABLED and caption_vf != "null":
            full_vf = f"{motion_vf},{caption_vf}"
        else:
            full_vf = motion_vf

        cmd = (
            f'ffmpeg -y -loop 1 -i "{img_path}" '
            f'-vf "{full_vf}" '
            f'-t {duration:.3f} -r {FPS} '
            f'-c:v libx264 -preset fast -crf 18 -pix_fmt yuv420p '
            f'"{effect_out}" -loglevel error'
        )
        ret = os.system(cmd)

        if ret != 0 or not os.path.exists(effect_out):
            # Fallback: static image without captions (always works)
            print(f"  WARNING: Effect failed for {sn}, trying static fallback...")
            fallback = (
                f'ffmpeg -y -loop 1 -i "{img_path}" '
                f'-vf "scale={WIDTH}:{HEIGHT},setsar=1" '
                f'-t {duration:.3f} -r {FPS} '
                f'-c:v libx264 -preset fast -crf 18 -pix_fmt yuv420p '
                f'"{effect_out}" -loglevel error'
            )
            ret2 = os.system(fallback)
            if ret2 != 0:
                print(f"  ERROR: Even static fallback failed for {sn}.")
                continue

        print(f"  {sn}.mp4 | {EFFECT_NAMES[idx % 6]} | {duration:.1f}s")

    print("--> Phase 3 complete.")


# ============================================================
# PHASE 4 — Per-clip Stitch: Effect Video + Voice Audio
# ============================================================
def run_phase_4_stitch():
    print("\n" + "="*60)
    print("PHASE 4: Per-clip Stitch  (Effect + Audio)")
    print("="*60)

    stitched_videos = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Stitching"):
        sn           = str(row["Serial number"])
        audio_len    = float(df.at[idx, "audio_length"])
        effect_video = os.path.join(EFFECT_DIR, f"{sn}.mp4")
        raw_audio    = os.path.join(AUDIO_DIR,  f"{sn}.wav")
        stitched_out = os.path.join(STITCH_DIR,  f"{sn}_clip.mp4")

        if os.path.exists(stitched_out):
            print(f"  Skipping {sn}_clip.mp4 (exists)")
            stitched_videos.append(stitched_out)
            continue

        missing = [p for p in [effect_video, raw_audio] if not os.path.exists(p)]
        if missing:
            print(f"  ERROR: Missing for {sn}: {missing}")
            continue

        cmd = (
            f'ffmpeg -y -stream_loop -1 -i "{effect_video}" -i "{raw_audio}" '
            f'-c:v libx264 -c:a aac -b:a 192k '
            f'-t {audio_len:.3f} -pix_fmt yuv420p '
            f'-shortest "{stitched_out}" -loglevel error'
        )
        ret = os.system(cmd)
        if ret == 0 and os.path.exists(stitched_out):
            stitched_videos.append(stitched_out)
            print(f"  {sn}_clip.mp4 ({audio_len:.1f}s) OK")
        else:
            print(f"  ERROR: Stitch failed for {sn}.")

    return stitched_videos


# ============================================================
# PHASE 5 — Concatenate + Speed Adjustment + BGM
#
# VIDEO_SPEED > 1.0  → faster (e.g. 1.25 = 25% faster)
# VIDEO_SPEED < 1.0  → slower (e.g. 0.9  = 10% slower)
# VIDEO_SPEED = 1.0  → no change
#
# Speed is applied BEFORE BGM so music always fills the
# exact final duration.
# ============================================================
def run_phase_5_final(stitched_videos):
    print("\n" + "="*60)
    print("PHASE 5: Final Assembly + Speed + BGM")
    print(f"         Speed: {VIDEO_SPEED}x")
    print("="*60)

    if not stitched_videos:
        raise RuntimeError("No stitched clips found — check Phases 1-4.")

    # ── Concatenate all clips (re-encode for guaranteed compatibility) ───────
    concat_txt = os.path.join(OUTPUT_DIR, "join_manifest.txt")
    with open(concat_txt, "w") as f:
        for v in stitched_videos:
            f.write(f"file '{v}'\n")
    print(f"  Concat manifest: {len(stitched_videos)} clips")

    pre_speed = os.path.join(OUTPUT_DIR, "master_raw.mp4")
    print("  Concatenating master timeline...")
    ret = os.system(
        f'ffmpeg -y -f concat -safe 0 -i "{concat_txt}" '
        f'-c:v libx264 -c:a aac -pix_fmt yuv420p '
        f'"{pre_speed}" -loglevel error'
    )
    if ret != 0 or not os.path.exists(pre_speed):
        raise RuntimeError("Master concat failed!")
    print(f"  Master timeline -> {pre_speed}")

    # ── Apply speed adjustment if needed ─────────────────────────────────────
    pre_bgm = pre_speed
    if abs(VIDEO_SPEED - 1.0) > 0.01:
        pre_bgm = os.path.join(OUTPUT_DIR, "master_sped.mp4")
        pts_mult  = round(1.0 / VIDEO_SPEED, 6)
        atempo    = build_atempo_chain(VIDEO_SPEED)
        print(f"  Applying {VIDEO_SPEED}x speed (pts={pts_mult}, atempo={atempo})...")
        speed_cmd = (
            f'ffmpeg -y -i "{pre_speed}" '
            f'-vf "setpts={pts_mult}*PTS" '
            f'-af "{atempo}" '
            f'-c:v libx264 -c:a aac -pix_fmt yuv420p '
            f'"{pre_bgm}" -loglevel error'
        )
        ret = os.system(speed_cmd)
        if ret != 0 or not os.path.exists(pre_bgm):
            print("  WARNING: Speed adjustment failed; using normal-speed master.")
            pre_bgm = pre_speed

    # ── BGM mix ──────────────────────────────────────────────────────────────
    final_out = os.path.join(
        OUTPUT_DIR,
        f"FINAL_AUTOMATED_OUTPUT_{ASPECT_RATIO.replace(':','_')}.mp4"
    )
    bgm_path = os.path.join(OUTPUT_DIR, "bgm.mp3")

    try:
        print("  Downloading background music...")
        urllib.request.urlretrieve(BGM_URL, bgm_path)
        bgm_cmd = (
            f'ffmpeg -y -i "{pre_bgm}" -stream_loop -1 -i "{bgm_path}" '
            f'-filter_complex "[1:a]volume=0.12[bgm];[0:a][bgm]amix=inputs=2:duration=first[aout]" '
            f'-map 0:v -map "[aout]" '
            f'-c:v copy -c:a aac -b:a 192k -shortest '
            f'"{final_out}" -loglevel error'
        )
        ret = os.system(bgm_cmd)
        if ret == 0 and os.path.exists(final_out):
            size_mb = os.path.getsize(final_out) / 1e6
            print(f"\n{'='*60}")
            print(f"  ✅ SUCCESS!  Final video ({size_mb:.1f} MB):")
            print(f"  {final_out}")
            print(f"{'='*60}")
        else:
            raise RuntimeError("BGM mix returned non-zero exit.")
    except Exception as e:
        print(f"  BGM injection failed ({e}). Saving without BGM.")
        shutil.copy(pre_bgm, final_out)
        size_mb = os.path.getsize(final_out) / 1e6
        print(f"\n{'='*60}")
        print(f"  ✅ SUCCESS (no BGM)!  Final video ({size_mb:.1f} MB):")
        print(f"  {final_out}")
        print(f"{'='*60}")


# ============================================================
# PIPELINE EXECUTION
# ============================================================
print("\n" + "#"*60)
print("# AUTOMATED VIDEO CHANNEL PIPELINE")
print("#  Phase 1 : Flux.1-Schnell  → Images")
print("#  Phase 2 : Kokoro TTS      → Voiceover Audio  (clean, vibrant)")
print("#  Phase 3 : FFmpeg          → Cinematic Motion + Captions")
print("#  Phase 4 : FFmpeg          → Per-clip Stitch")
print("#  Phase 5 : FFmpeg          → Concat + Speed + BGM")
print("#" + "="*58)
print(f"#  Entries: {len(df)}  |  {ASPECT_RATIO}  |  {WIDTH}x{HEIGHT}")
print(f"#  Voice: {KOKORO_VOICE}  |  Speed: {VIDEO_SPEED}x")
print(f"#  Captions: {CAPTION_ENABLED}  |  Font: {CAPTION_FONT_SIZE}px")
print("#"*60 + "\n")

if USE_Z_IMAGE:
    run_phase_1_z_image()
else:
    run_phase_1_flux()
force_clear_memory()

run_phase_2_audio()
force_clear_memory()

run_phase_3_effects()

stitched = run_phase_4_stitch()
run_phase_5_final(stitched)

print("\n--> Full pipeline complete.")
print(f"--> All outputs in: {OUTPUT_DIR}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


--> Settings: 16:9 | 736x416 | FPS:24 | Speed:1.1x
    Voice: am_michael | Captions: True | Font size: 22px

--> Loading input dataset...
    Loaded from Google Drive: 15 entries.

############################################################
# AUTOMATED VIDEO CHANNEL PIPELINE
#  Phase 1 : Flux.1-Schnell  → Images
#  Phase 2 : Kokoro TTS      → Voiceover Audio  (clean, vibrant)
#  Phase 3 : FFmpeg          → Cinematic Motion + Captions
#  Phase 4 : FFmpeg          → Per-clip Stitch
#  Phase 5 : FFmpeg          → Concat + Speed + BGM
#==========================================================
#  Entries: 15  |  16:9  |  736x416
#  Voice: am_michael  |  Speed: 1.1x
#  Captions: True  |  Font: 22px
############################################################


PHASE 1: Image Generation  (Flux.1-Schnell)
  [INFO] Kaggle Secrets not available (BackendError) — that is OK.
--> HF authenticated via config override as: Airpyk98


model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

Stitching:  93%|█████████▎| 14/15 [00:35<00:02,  2.84s/it]

  14_clip.mp4 (15.4s) OK


Stitching: 100%|██████████| 15/15 [00:38<00:00,  2.53s/it]

  15_clip.mp4 (15.7s) OK

PHASE 5: Final Assembly + Speed + BGM
         Speed: 1.1x
  Concat manifest: 15 clips
  Concatenating master timeline...


  Master timeline -> /kaggle/working/automated_channel_outputs/master_raw.mp4
  Applying 1.1x speed (pts=0.909091, atempo=atempo=1.100000)...

  ✅ SUCCESS!  Final video (10.5 MB):
  /kaggle/working/automated_channel_outputs/FINAL_AUTOMATED_OUTPUT_16_9.mp4

--> Full pipeline complete.
--> All outputs in: /kaggle/working/automated_channel_outputs
